Teste para ver se existe a tabela ou se terei que criar uma nova

In [0]:
def tabela_existe(spark, jdbc_url, connection_properties, schema, tabela):
    query = f"""
    (
        SELECT COUNT(*) AS qtd
        FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = '{schema}'
          AND TABLE_NAME = '{tabela}'
    ) x
    """

    qtd = (
        spark.read.jdbc(
            url=jdbc_url,
            table=query,
            properties=connection_properties
        )
        .collect()[0]["qtd"]
    )

    return qtd > 0


def escrever_sqlserver(
    df_spark,
    spark,
    jdbc_url,
    connection_properties,
    jdbc_hostname,
    jdbc_database,
    jdbc_username,
    jdbc_password,
    schema,
    tabela
):
    tabela_destino = f"{schema}.{tabela}"

    if tabela_existe(spark, jdbc_url, connection_properties, schema, tabela):
        modo = "append"
        print(f"Tabela {tabela_destino} já existe. Inserindo novos dados.")
    else:
        modo = "overwrite"
        print(f"Tabela {tabela_destino} não existe. Criando tabela.")

    (
        df_spark.write
        .format("sqlserver")
        .mode(modo)
        .option("host", jdbc_hostname)
        .option("port", "1433")
        .option("database", jdbc_database)
        .option("user", jdbc_username)
        .option("password", jdbc_password)
        .option("dbtable", tabela_destino)
        .option("encrypt", "true")
        .option("trustServerCertificate", "false")
        .save()
    )

    print(f"Carga finalizada em {tabela_destino}. Modo usado: {modo}")